# Task 1: Compare document-processing approaches

Process the same two-page claim statement with three pipelines and compare their normalized results against supplied ground truth.

| Pipeline | Document processing | Structuring |
|---|---|---|
| Configured multimodal GPT model | Reads both images directly | Structured output |
| Mistral Document AI | OCR to Markdown | Configured GPT model normalizes the OCR text |
| Azure Document Intelligence | Prebuilt layout extraction | Configured GPT model normalizes the extracted text |

Using the same normalizer for both OCR pipelines makes their output comparable while preserving their raw text for inspection.

## 1. Load configuration

Select the repository's Python environment as the notebook kernel. The `.env` file must have been generated in Challenge 0.

In [ ]:
from __future__ import annotations

import base64
import json
import os
import re
import sys
import time
from difflib import SequenceMatcher
from pathlib import Path
from typing import Optional

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Image, Markdown, display
from pydantic import BaseModel, ConfigDict


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "challenge-0").is_dir() and (candidate / "challenge-1").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from inside the claims-processing-hack repository.")


REPO_ROOT = find_repo_root()
load_dotenv(REPO_ROOT / ".env")

CLAIM_ID = "crash1"
STATEMENTS_DIR = REPO_ROOT / "challenge-0" / "data" / "statements"
FRONT_IMAGE = STATEMENTS_DIR / f"{CLAIM_ID}_front.jpeg"
BACK_IMAGE = STATEMENTS_DIR / f"{CLAIM_ID}_back.jpeg"
OUTPUT_DIR = REPO_ROOT / "challenge-1" / "output" / "comparison" / CLAIM_ID
GROUND_TRUTH_FILE = REPO_ROOT / "challenge-1" / "comparison_ground_truth.json"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

required_variables = [
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_KEY",
    "AZURE_OPENAI_DEPLOYMENT_NAME",
    "AZURE_OPENAI_API_VERSION",
    "MISTRAL_DOCUMENT_AI_ENDPOINT",
    "MISTRAL_DOCUMENT_AI_KEY",
    "MISTRAL_DOCUMENT_AI_DEPLOYMENT_NAME",
    "DOCUMENT_INTELLIGENCE_ENDPOINT",
    "DOCUMENT_INTELLIGENCE_KEY",
]
missing_variables = [name for name in required_variables if not os.getenv(name)]
if missing_variables:
    raise RuntimeError(f"Missing variables in .env: {', '.join(missing_variables)}")

print(f"Repository: {REPO_ROOT}")
print(f"Claim: {CLAIM_ID}")
print(f"Multimodal/normalization deployment: {os.environ['AZURE_OPENAI_DEPLOYMENT_NAME']}")
print(f"Output directory: {OUTPUT_DIR}")

## 2. Inspect the source document

Review both pages before looking at model output. This establishes what a correct extraction should contain.

In [ ]:
display(Markdown("### Front page"))
display(Image(filename=str(FRONT_IMAGE), width=700))
display(Markdown("### Back page"))
display(Image(filename=str(BACK_IMAGE), width=700))

## 3. Define one shared output schema

Every pipeline must return the same fields. Missing or unreadable values remain `null`; models must not invent them.

In [ ]:
class ClaimStatement(BaseModel):
    model_config = ConfigDict(extra="forbid")

    claimant_id: Optional[str] = None
    policy_holder_name: Optional[str] = None
    policy_holder_address: Optional[str] = None
    policy_holder_phone: Optional[str] = None
    policy_holder_email: Optional[str] = None
    policy_number: Optional[str] = None
    vehicle_year_make_model: Optional[str] = None
    vehicle_color: Optional[str] = None
    vehicle_vin: Optional[str] = None
    vehicle_license_plate: Optional[str] = None
    incident_date: Optional[str] = None
    incident_time: Optional[str] = None
    incident_location: Optional[str] = None
    incident_description: Optional[str] = None
    damage_description: Optional[str] = None
    witness_name: Optional[str] = None
    witness_phone: Optional[str] = None
    police_department: Optional[str] = None
    police_report_number: Optional[str] = None
    officer_name: Optional[str] = None
    repair_shop_name: Optional[str] = None
    repair_shop_address: Optional[str] = None
    claim_request: Optional[str] = None
    signature_name: Optional[str] = None
    signature_date: Optional[str] = None


EXTRACTION_INSTRUCTIONS = """
Extract the insurance claim statement into the supplied schema.
Use only information explicitly present in the document.
Combine information from the front and back pages.
Preserve identifiers, names, dates, phone numbers, addresses, VINs, and license plates accurately.
Use null for missing, illegible, or uncertain fields. Do not infer or invent values.
""".strip()

## 4. Configure the shared structured-output helper

In [ ]:
from openai import AzureOpenAI

openai_client = AzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_KEY"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
)


def image_data_url(path: Path) -> str:
    encoded = base64.b64encode(path.read_bytes()).decode("utf-8")
    return f"data:image/jpeg;base64,{encoded}"


def parse_claim(messages: list[dict]) -> ClaimStatement:
    completion = openai_client.beta.chat.completions.parse(
        model=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
        messages=messages,
        response_format=ClaimStatement,
        max_tokens=4000,
    )
    parsed = completion.choices[0].message.parsed
    if parsed is None:
        refusal = completion.choices[0].message.refusal
        raise RuntimeError(f"The model did not return a claim: {refusal}")
    return parsed


def normalize_ocr_text(raw_text: str, source_name: str) -> ClaimStatement:
    return parse_claim([
        {
            "role": "system",
            "content": EXTRACTION_INSTRUCTIONS,
        },
        {
            "role": "user",
            "content": f"Normalize this {source_name} OCR output:\n\n{raw_text}",
        },
    ])

## 5. Pipeline A: direct multimodal extraction

The configured GPT deployment sees both images and returns the shared schema directly.

In [ ]:
started = time.perf_counter()
gpt_result = parse_claim([
    {
        "role": "system",
        "content": EXTRACTION_INSTRUCTIONS,
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Extract this two-page claim statement."},
            {"type": "image_url", "image_url": {"url": image_data_url(FRONT_IMAGE)}},
            {"type": "image_url", "image_url": {"url": image_data_url(BACK_IMAGE)}},
        ],
    },
])
gpt_seconds = time.perf_counter() - started
display(pd.Series(gpt_result.model_dump(), name="GPT direct").to_frame())
print(f"Completed in {gpt_seconds:.2f} seconds")

## 6. Pipeline B: Mistral Document AI OCR

Mistral processes each page independently. The raw Markdown is retained, then normalized into the shared schema.

In [ ]:
statements_processing_dir = REPO_ROOT / "challenge-1" / "statements_processing"
if str(statements_processing_dir) not in sys.path:
    sys.path.insert(0, str(statements_processing_dir))

from mistral_doc_intelligence import get_ocr_results

started = time.perf_counter()
mistral_front = get_ocr_results(str(FRONT_IMAGE))
mistral_back = get_ocr_results(str(BACK_IMAGE))
mistral_raw = f"# Front page\n\n{mistral_front}\n\n# Back page\n\n{mistral_back}"
mistral_result = normalize_ocr_text(mistral_raw, "Mistral Document AI")
mistral_seconds = time.perf_counter() - started

display(Markdown(mistral_raw))
display(pd.Series(mistral_result.model_dump(), name="Mistral + normalizer").to_frame())
print(f"Completed in {mistral_seconds:.2f} seconds")

## 7. Pipeline C: Azure Document Intelligence

The `prebuilt-layout` model extracts Markdown from each page. The same normalizer used for Mistral converts it into the shared schema.

In [ ]:
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import DocumentContentFormat
from azure.core.credentials import AzureKeyCredential

document_intelligence_client = DocumentIntelligenceClient(
    endpoint=os.environ["DOCUMENT_INTELLIGENCE_ENDPOINT"].rstrip("/"),
    credential=AzureKeyCredential(os.environ["DOCUMENT_INTELLIGENCE_KEY"]),
)


def analyze_layout(path: Path) -> str:
    with path.open("rb") as document:
        poller = document_intelligence_client.begin_analyze_document(
            "prebuilt-layout",
            body=document,
            content_type="image/jpeg",
            output_content_format=DocumentContentFormat.MARKDOWN,
        )
    return poller.result().content


started = time.perf_counter()
document_intelligence_front = analyze_layout(FRONT_IMAGE)
document_intelligence_back = analyze_layout(BACK_IMAGE)
document_intelligence_raw = (
    f"# Front page\n\n{document_intelligence_front}"
    f"\n\n# Back page\n\n{document_intelligence_back}"
)
document_intelligence_result = normalize_ocr_text(
    document_intelligence_raw,
    "Azure Document Intelligence",
)
document_intelligence_seconds = time.perf_counter() - started

display(Markdown(document_intelligence_raw))
display(pd.Series(
    document_intelligence_result.model_dump(),
    name="Document Intelligence + normalizer",
).to_frame())
print(f"Completed in {document_intelligence_seconds:.2f} seconds")

## 8. Compare fields and scores

The similarity score normalizes punctuation and spacing before comparing text. Values returned for fields that should be empty are counted as unsupported extractions. Treat the automatic score as a guide and inspect consequential fields manually.

This is a single-sample workshop comparison, not a production benchmark. Mistral and Document Intelligence also process the two pages sequentially before normalization, while the direct GPT pipeline uses one request, so elapsed time is not a pure model-speed measurement.

In [ ]:
ground_truth = json.loads(GROUND_TRUTH_FILE.read_text(encoding="utf-8"))[CLAIM_ID]
pipeline_results = {
    "GPT direct": gpt_result.model_dump(),
    "Mistral + normalizer": mistral_result.model_dump(),
    "Document Intelligence + normalizer": document_intelligence_result.model_dump(),
}


def normalize_for_comparison(value: Optional[str]) -> str:
    if value is None:
        return ""
    return re.sub(r"[^a-z0-9]+", " ", str(value).lower()).strip()


def similarity(expected: Optional[str], actual: Optional[str]) -> float:
    expected_normalized = normalize_for_comparison(expected)
    actual_normalized = normalize_for_comparison(actual)
    if not expected_normalized:
        return 1.0 if not actual_normalized else 0.0
    if not actual_normalized:
        return 0.0
    return SequenceMatcher(None, expected_normalized, actual_normalized).ratio()


comparison_rows = []
for field, expected in ground_truth.items():
    row = {"Field": field, "Ground truth": expected}
    for pipeline, result in pipeline_results.items():
        row[pipeline] = result.get(field)
    comparison_rows.append(row)

field_comparison = pd.DataFrame(comparison_rows).set_index("Field")
display(field_comparison)

latencies = {
    "GPT direct": gpt_seconds,
    "Mistral + normalizer": mistral_seconds,
    "Document Intelligence + normalizer": document_intelligence_seconds,
}
expected_fields = [field for field, expected in ground_truth.items() if expected is not None]
identifier_fields = [
    "claimant_id",
    "policy_number",
    "vehicle_vin",
    "vehicle_license_plate",
    "policy_holder_phone",
    "policy_holder_email",
    "police_report_number",
]
summary_rows = []
for pipeline, result in pipeline_results.items():
    scores = [similarity(expected, result.get(field)) for field, expected in ground_truth.items()]
    populated = sum(bool(normalize_for_comparison(result.get(field))) for field in expected_fields)
    unsupported = sum(
        expected is None and bool(normalize_for_comparison(result.get(field)))
        for field, expected in ground_truth.items()
    )
    identifier_matches = sum(
        normalize_for_comparison(ground_truth[field]) == normalize_for_comparison(result.get(field))
        for field in identifier_fields
    )
    summary_rows.append({
        "Pipeline": pipeline,
        "Mean field similarity": sum(scores) / len(scores),
        "Identifier exact match": identifier_matches / len(identifier_fields),
        "Fields populated": populated,
        "Expected fields": len(expected_fields),
        "Completeness": populated / len(expected_fields),
        "Unsupported extractions": unsupported,
        "Elapsed seconds": latencies[pipeline],
    })

summary = pd.DataFrame(summary_rows).set_index("Pipeline")
display(summary.style.format({
    "Mean field similarity": "{:.1%}",
    "Identifier exact match": "{:.1%}",
    "Completeness": "{:.1%}",
    "Elapsed seconds": "{:.2f}",
}))

## 9. Export clean and readable artifacts

In [ ]:
def markdown_table(frame: pd.DataFrame) -> str:
    printable = frame.reset_index().fillna("").astype(str)
    headers = list(printable.columns)
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join("---" for _ in headers) + " |",
    ]
    for row in printable.itertuples(index=False, name=None):
        values = [value.replace("|", "\\|").replace("\n", " ") for value in row]
        lines.append("| " + " | ".join(values) + " |")
    return "\n".join(lines)


for pipeline, result in pipeline_results.items():
    file_name = pipeline.lower().replace(" + ", "_").replace(" ", "_") + ".json"
    (OUTPUT_DIR / file_name).write_text(
        json.dumps(result, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

(OUTPUT_DIR / "mistral_raw.md").write_text(mistral_raw, encoding="utf-8")
(OUTPUT_DIR / "document_intelligence_raw.md").write_text(
    document_intelligence_raw,
    encoding="utf-8",
)
(OUTPUT_DIR / "summary.json").write_text(
    summary.reset_index().to_json(orient="records", indent=2),
    encoding="utf-8",
)

report = f"""# Statement-processing comparison: {CLAIM_ID}

## Summary

{markdown_table(summary)}

## Field comparison

{markdown_table(field_comparison)}
"""
(OUTPUT_DIR / "comparison.md").write_text(report, encoding="utf-8")

print(f"Artifacts written to {OUTPUT_DIR}")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f"- {path.name}")

## 10. Make a recommendation

Discuss these questions with your group:

1. Which pipeline was most accurate on identifiers such as policy number, VIN, phone number, and police report number?
2. Which pipeline best preserved the longer incident and damage descriptions?
3. Were any values invented or silently corrected by the normalizer?
4. How much latency did the OCR-plus-normalizer pipelines add?
5. Which raw output would be most useful for auditability and human review?
6. Which approach would you choose for high-volume standardized forms, and which for complex handwritten statements?

**Success criterion:** all three normalized results and the summary table exist, and your group can defend a model-selection decision using the observed evidence.